In [2]:
import os
patches_path = "/content/drive/MyDrive/TerraSpectra/data/patches"
print("Mere patches folder mein yeh files hain:", os.listdir(patches_path))

Mere patches folder mein yeh files hain: ['terraspectra_patches.npz']


In [3]:
import numpy as np

npz_path = "/content/drive/MyDrive/TerraSpectra/data/patches/terraspectra_patches.npz"
data = np.load(npz_path)

print("Keys inside .npz:", data.files)
for k in data.files:
    print(f"  {k}: shape={data[k].shape}, dtype={data[k].dtype}")

Keys inside .npz: ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']
  X_train: shape=(7174, 15, 15, 30), dtype=float32
  y_train: shape=(7174,), dtype=int64
  X_val: shape=(1537, 15, 15, 30), dtype=float32
  y_val: shape=(1537,), dtype=int64
  X_test: shape=(1538, 15, 15, 30), dtype=float32
  y_test: shape=(1538,), dtype=int64


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 1. GPU Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Data Extract aur Shape Set karna
# PyTorch 3D Conv ko (Batch, 1, 30, 15, 15) shape chahiye hota hai
def prepare_data(X, y, batch_size=64, shuffle=False):
    X_t = np.transpose(X, (0, 3, 1, 2))   # (N, 30, 15, 15)
    X_t = np.expand_dims(X_t, axis=1)     # (N, 1, 30, 15, 15)
    ds = TensorDataset(torch.tensor(X_t, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = prepare_data(data["X_train"], data["y_train"], batch_size=64, shuffle=True)
val_loader   = prepare_data(data["X_val"],   data["y_val"],   batch_size=64, shuffle=False)
test_loader  = prepare_data(data["X_test"],  data["y_test"],  batch_size=64, shuffle=False)

num_classes = len(np.unique(data["y_train"]))
print(f"Total Classes: {num_classes} | Min label: {data['y_train'].min()} | Max label: {data['y_train'].max()}")

# 3. HybridSN Model (3D + 2D CNN)
class HybridSN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3)),
            nn.BatchNorm3d(8),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3)),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
        )
        self.conv2d = nn.Sequential(
            nn.Conv2d(32 * 18, 64, kernel_size=(3, 3)),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.conv3d(x)
        b, c, d, h, w = x.shape
        x = x.view(b, c * d, h, w)
        x = self.conv2d(x)
        return self.classifier(x)

model = HybridSN(num_classes=num_classes).to(device)
print("HybridSN Model ready!")

Using device: cuda
Total Classes: 16 | Min label: 0 | Max label: 15
HybridSN Model ready!


In [5]:
import os
import torch.optim as optim

# Save path aapke existing artifacts folder mein set kar rahe hain
SAVE_DIR = "/content/drive/MyDrive/TerraSpectra/data/artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = f"{SAVE_DIR}/hybridsn_indian_pines.pth"

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

best_val_acc = 0.0

for epoch in range(1, 51):
    # --- Training ---
    model.train()
    correct, total = 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        outputs = model(X_b)
        loss = criterion(outputs, y_b)
        loss.backward()
        optimizer.step()

        correct += (outputs.argmax(1) == y_b).sum().item()
        total += y_b.size(0)
    train_acc = (correct / total) * 100

    # --- Validation ---
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model(X_b).argmax(1)
            val_correct += (preds == y_b).sum().item()
            val_total += y_b.size(0)
    val_acc = (val_correct / val_total) * 100
    scheduler.step(val_acc)

    # Best model save karna
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/50 | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Best Val: {best_val_acc:.2f}%")

print(f"\nTraining Complete! Best model saved at: {MODEL_PATH}")

Epoch 01/50 | Train Acc: 74.24% | Val Acc: 95.64% | Best Val: 95.64%
Epoch 05/50 | Train Acc: 98.83% | Val Acc: 99.09% | Best Val: 99.67%
Epoch 10/50 | Train Acc: 99.39% | Val Acc: 99.87% | Best Val: 99.93%
Epoch 15/50 | Train Acc: 99.82% | Val Acc: 100.00% | Best Val: 100.00%
Epoch 20/50 | Train Acc: 99.86% | Val Acc: 99.93% | Best Val: 100.00%
Epoch 25/50 | Train Acc: 99.94% | Val Acc: 100.00% | Best Val: 100.00%
Epoch 30/50 | Train Acc: 99.97% | Val Acc: 100.00% | Best Val: 100.00%
Epoch 35/50 | Train Acc: 99.97% | Val Acc: 100.00% | Best Val: 100.00%
Epoch 40/50 | Train Acc: 99.96% | Val Acc: 100.00% | Best Val: 100.00%
Epoch 45/50 | Train Acc: 99.97% | Val Acc: 100.00% | Best Val: 100.00%
Epoch 50/50 | Train Acc: 99.97% | Val Acc: 100.00% | Best Val: 100.00%

Training Complete! Best model saved at: /content/drive/MyDrive/TerraSpectra/data/artifacts/hybridsn_indian_pines.pth


In [6]:
import json
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

# Saved best model wapas load karna
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        preds = model(X_b).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_b.numpy())

test_oa = accuracy_score(all_targets, all_preds) * 100
kappa = cohen_kappa_score(all_targets, all_preds)

print(f"=== FINAL TEST RESULTS ===")
print(f"Test Overall Accuracy (OA): {test_oa:.2f}%")
print(f"Cohen's Kappa Score:        {kappa:.4f}\n")
print(classification_report(all_targets, all_preds, digits=4))

# Metrics ko artifacts folder mein save karna
metrics = {
    "model": "HybridSN (3D-2D CNN)",
    "dataset": "Indian Pines (15x15x30 patches)",
    "test_overall_accuracy": round(test_oa, 2),
    "cohen_kappa": round(float(kappa), 4),
    "num_classes": int(num_classes),
    "test_samples": len(all_targets)
}

metrics_path = f"{SAVE_DIR}/metrics_day6.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved at: {metrics_path}")


=== FINAL TEST RESULTS ===
Test Overall Accuracy (OA): 99.87%
Cohen's Kappa Score:        0.9985

              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000         7
           1     0.9953    1.0000    0.9977       214
           2     1.0000    1.0000    1.0000       125
           3     1.0000    1.0000    1.0000        35
           4     1.0000    1.0000    1.0000        73
           5     0.9910    1.0000    0.9955       110
           6     1.0000    1.0000    1.0000         4
           7     1.0000    1.0000    1.0000        71
           8     1.0000    1.0000    1.0000         3
           9     1.0000    1.0000    1.0000       146
          10     1.0000    0.9946    0.9973       369
          11     1.0000    1.0000    1.0000        89
          12     1.0000    1.0000    1.0000        30
          13     1.0000    1.0000    1.0000       190
          14     1.0000    1.0000    1.0000        58
          15     1.0000    1.0000    